# Ingest lap_times folder


In [0]:
%run "../includes/common_functions"

In [0]:
%run "../includes/configuration"

In [0]:
dbutils.widgets.text("p_date_source","")
#dbutils.widgets.dropdown("p_date_source","Testing",["Testing","Production"])
v_data_source=dbutils.widgets.get("p_date_source")

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType,DateType
from pyspark.sql.functions import current_timestamp,to_date,current_date,lit,to_timestamp,concat,col


In [0]:
lap_times_schema = StructType(fields=[StructField("raceId", IntegerType(), False),StructField("driverId", IntegerType(), True),StructField("lap", IntegerType(), True),StructField("position", IntegerType(), True),StructField("time", StringType(), True),StructField("milliseconds", StringType(), True)])
lap_times_df = spark.read.schema(lap_times_schema).csv(f"{raw_folder_path}/lap_times")
#lap_times_df = spark.read.schema(lap_times_schema).csv(f"{raw_folder_path}/lap_times/lap_times_split*.csv")

In [0]:
lap_times_df = lap_times_df.withColumnRenamed('raceId','race_id').withColumnRenamed('driverId','driver_id')

In [0]:
lap_times_df=add_ingestion_timestamp(lap_times_df)
lap_times_df = add_data_source(lap_times_df,v_data_source)

In [0]:
#display(lap_times_df)
#lap_times_df.count()

## Write DF into parquet file

In [0]:
lap_times_df.write.mode("overwrite").partitionBy("race_id").parquet(f"{processed_folder_path}/lap_times")

In [0]:
df=spark.read.parquet(f"{processed_folder_path}/lap_times")
df.printSchema()

In [0]:
dbutils.notebook.exit("Success")